<a href="https://colab.research.google.com/github/jingle77/Data-Science-Portfolio/blob/main/MNIST_SOLUTION_NO_TENSORFLOW.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Import Libraries

In [35]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_digits
from sklearn.preprocessing import StandardScaler

## Import Data

In [36]:
## Import test and train data
mnist = load_digits()
mnist_features = pd.DataFrame(mnist.data)
mnist_target = pd.DataFrame(mnist.target)

In [37]:
## Data has already been flattened
## In other words, the 8 by 8 pixel grid has been flattened into 64 columns for the features
mnist_features.shape

(1797, 64)

In [38]:
## Combine into 1 df
mnist_features['target'] = mnist_target.values
mnist_df = mnist_features

## Shuffle 100% of the data as there appears to be a pre defined order
shuffled_df = mnist_df.sample(frac=1).reset_index(drop = True)

## Process data for training

In [39]:
## Converting to an array so we can perform linear algebra on our dataset
df_array = np.array(shuffled_df)

train = df_array[:1000].T
test = df_array[1000:].T

In [40]:
## Dimensions
m,n = df_array.shape

In [41]:
## Features and Targets for train and test data sets
X_train = train[0:(n-1)]
Y_train = train[n-1]
Y_train = Y_train.astype(int)


X_test = test[0:(n-1)]
Y_test = test[n-1]
Y_test = Y_test.astype(int)

## Build and Evaluate Model

#### Functions

In [43]:
def init_params():
    W1 = np.random.rand(10, 64) - 0.5
    b1 = np.random.rand(10, 1) - 0.5
    W2 = np.random.rand(10, 10) - 0.5
    b2 = np.random.rand(10, 1) - 0.5
    return W1, b1, W2, b2

def ReLU(Z):
    return np.maximum(Z, 0)

def softmax(Z):
    A = np.exp(Z) / sum(np.exp(Z))
    return A
    
def forward_prop(W1, b1, W2, b2, X):
    Z1 = W1.dot(X) + b1
    A1 = ReLU(Z1)
    Z2 = W2.dot(A1) + b2
    A2 = softmax(Z2)
    return Z1, A1, Z2, A2

def ReLU_deriv(Z):
    return Z > 0

def one_hot(Y):
    one_hot_Y = np.zeros((Y.size, Y.max() + 1))
    one_hot_Y[np.arange(Y.size), Y] = 1
    one_hot_Y = one_hot_Y.T
    return one_hot_Y

def backward_prop(Z1, A1, Z2, A2, W1, W2, X, Y):
    one_hot_Y = one_hot(Y)
    dZ2 = A2 - one_hot_Y
    dW2 = 1 / m * dZ2.dot(A1.T)
    db2 = 1 / m * np.sum(dZ2)
    dZ1 = W2.T.dot(dZ2) * ReLU_deriv(Z1)
    dW1 = 1 / m * dZ1.dot(X.T)
    db1 = 1 / m * np.sum(dZ1)
    return dW1, db1, dW2, db2

def update_params(W1, b1, W2, b2, dW1, db1, dW2, db2, alpha):
    W1 = W1 - alpha * dW1
    b1 = b1 - alpha * db1    
    W2 = W2 - alpha * dW2  
    b2 = b2 - alpha * db2    
    return W1, b1, W2, b2

In [59]:
def get_predictions(A2):
    return np.argmax(A2, 0)

def get_accuracy(predictions, Y):
    return np.sum(predictions == Y) / Y.size

def gradient_descent(X, Y, alpha, iterations):
    W1, b1, W2, b2 = init_params()
    for i in range(iterations):
        Z1, A1, Z2, A2 = forward_prop(W1, b1, W2, b2, X)
        dW1, db1, dW2, db2 = backward_prop(Z1, A1, Z2, A2, W1, W2, X, Y)
        W1, b1, W2, b2 = update_params(W1, b1, W2, b2, dW1, db1, dW2, db2, alpha)
        if i % 100 == 0:
            print("Iteration: ", i)
            predictions = get_predictions(A2)
            print(get_accuracy(predictions, Y))
    return W1, b1, W2, b2

In [60]:
def make_predictions(X, W1, b1, W2, b2):
    _, _, _, A2 = forward_prop(W1, b1, W2, b2, X)
    predictions = get_predictions(A2)
    return predictions

#### Run Model

In [61]:
W1, b1, W2, b2 = gradient_descent(X_train, Y_train, 0.10, 1000)

Iteration:  0
0.089
Iteration:  100
0.622
Iteration:  200
0.724
Iteration:  300
0.811
Iteration:  400
0.907
Iteration:  500
0.919
Iteration:  600
0.923
Iteration:  700
0.939
Iteration:  800
0.943
Iteration:  900
0.95


#### Evaluate Model

In [62]:
test_predictions = make_predictions(X_test, W1, b1, W2, b2)
get_accuracy(test_predictions, Y_test)

0.8946047678795483